# Minimal cross-store query — Netherlands 2024

Pull **CO2** from the ICOS Obspack zarr, **monthly NEE** from the
FLUXNET zarr, and **fCO₂** from the ICOS Ocean (SOCAT) zarr for
stations / cruises inside a Netherlands bounding box, restricted to
2024.

- **Region**: lat 50.7–53.6, lon 3.3–7.3
- **Time**: 2024-01-01 → 2024-12-31

Each store has a different shape:

| Store | Group layout | Spatial coords | Query pattern |
|---|---|---|---|
| `icos-obspack.zarr` | combined-view: `co2`, `ch4`, `n2o`, `co` | `lat`, `lon`, `intake_height` | one xarray expression with `.where(mask, drop=True)` |
| `icos-fluxnet.zarr` | combined-view: `_combined/fluxnet_dd` … `_yy` | `lat`, `lon`, `station_elevation` | same pattern |
| `icos-socat.zarr` | per-cruise groups (one per CSV) | `lon(time)`, `lat(time)` | small `for`-loop over cruise IDs from `.zmetadata` |

The combined-view groups (built by `combine_to_dim.py`) make spatial
selection a single xarray expression for Obspack and Fluxnet.  SOCAT's
per-cruise layout reflects its source structure (one CSV per cruise leg
or buoy deployment), so we iterate.

In [ ]:
import xarray as xr

LAT_MIN, LAT_MAX = 50.7, 53.6
LON_MIN, LON_MAX = 3.3, 7.3
T0, T1 = "2024-01-01", "2024-12-31"

## Obspack — CO2

In [ ]:
ds = xr.open_zarr("zarr//icos-obspack.zarr", group="co2", consolidated=False)

co2_nl = (
    ds["co2"]
      .where(
          (ds.lat >= LAT_MIN) & (ds.lat <= LAT_MAX) &
          (ds.lon >= LON_MIN) & (ds.lon <= LON_MAX),
          drop=True,
      )
      .sel(time_co2=slice(T0, T1))
)

# Per-station summary — coords travel with the DataArray
for sid in co2_nl.station.values:
    da = co2_nl.sel(station=sid)
    print(f"{sid:8s} lat={float(co2_nl.lat.sel(station=sid)):.3f} "
          f"lon={float(co2_nl.lon.sel(station=sid)):.3f}  "
          f"intake_height={float(co2_nl.intake_height.sel(station=sid)):>4.0f} m  "
          f"n={int(da.count())}  "
          f"mean={float(da.mean()):.2f} ppm")

## FLUXNET — monthly NEE

Selects the VUT/REF NEE variant from each site's `fluxnet_mm` sub-group.

In [ ]:
ds = xr.open_zarr("zarr//icos-fluxnet.zarr", group="_combined/fluxnet_mm", consolidated=False)

nee_nl = (
    ds["NEE"]
      .sel(ustar_threshold="VUT", nee_variant="REF")
      .where(
          (ds.lat >= LAT_MIN) & (ds.lat <= LAT_MAX) &
          (ds.lon >= LON_MIN) & (ds.lon <= LON_MAX),
          drop=True,
      )
      .sel(time=slice(T0, T1))
)

units = ds["NEE"].attrs.get("units", "")
for sid in nee_nl.station.values:
    da = nee_nl.sel(station=sid)
    print(f"{sid:8s} lat={float(nee_nl.lat.sel(station=sid)):.3f} "
          f"lon={float(nee_nl.lon.sel(station=sid)):.3f}  "
          f"n={int(da.count())}  "
          f"mean={float(da.mean()):.3f} {units}")

## SOCAT — fCO₂ in Dutch / nearby waters

The SOCAT store has a different shape from Obspack/Fluxnet — one zarr
group per cruise leg or buoy deployment, not a combined-view group with
a `station` dim.  So the query is a small loop: enumerate the cruises
from the consolidated `.zmetadata`, open each one, and select the rows
that fall inside the bounding box and time window.  The same WOCE QC
mask the ENVRI app applies (flag ≤ 2 on `fCO2_QC`, `lon_QC`, `lat_QC`)
is applied here too.

In [ ]:
import json
import pathlib
import numpy as np

SOCAT_STORE = pathlib.Path("zarr//icos-socat.zarr")

with open(SOCAT_STORE / ".zmetadata") as fh:
    socat_meta = json.load(fh).get("metadata") or {}

cruises = sorted({
    k.split("/", 1)[0]
    for k in socat_meta
    if k.endswith("/.zgroup") and k.count("/") == 1
    and not k.startswith(".") and not k.startswith("_")
})

per_cruise: list[dict] = []
for cruise_id in cruises:
    ds = xr.open_zarr(SOCAT_STORE / cruise_id, consolidated=True)
    try:
        if "lon" not in ds or "lat" not in ds or "fCO2" not in ds:
            continue   # fixed buoys without lon/lat or stores missing fCO2
        lon  = ds["lon"].values.astype(float)
        lat  = ds["lat"].values.astype(float)
        fco2 = ds["fCO2"].values.astype(float)
        time = ds["time"].values

        ok = (
            np.isfinite(lon) & np.isfinite(lat) & np.isfinite(fco2)
            & (lat >= LAT_MIN) & (lat <= LAT_MAX)
            & (lon >= LON_MIN) & (lon <= LON_MAX)
            & (time >= np.datetime64(T0)) & (time <= np.datetime64(T1))
        )
        for qc in ("fCO2_QC", "lon_QC", "lat_QC"):
            if qc in ds:
                v = ds[qc].values
                ok &= (v >= 0) & (v <= 2)

        n = int(ok.sum())
        if n:
            per_cruise.append({
                "cruise":  cruise_id,
                "station": ds.attrs.get("station_id") or ds.attrs.get("platform_name"),
                "n":       n,
                "fco2_mean": float(fco2[ok].mean()),
                "fco2_min":  float(fco2[ok].min()),
                "fco2_max":  float(fco2[ok].max()),
            })
    finally:
        ds.close()

print(f"SOCAT — {len(per_cruise)} cruise(s) with QC-passing samples in NL+2024:\n")
for c in per_cruise:
    print(f"  {c['cruise']:14s}  {c['station'] or '?':30s}  "
          f"n={c['n']:>5d}  fCO2 mean={c['fco2_mean']:.1f}  "
          f"({c['fco2_min']:.1f}–{c['fco2_max']:.1f}) µatm")

if per_cruise:
    total_n = sum(c["n"] for c in per_cruise)
    pooled_mean = sum(c["fco2_mean"] * c["n"] for c in per_cruise) / total_n
    print(f"\nPooled mean fCO2 across {total_n} samples = {pooled_mean:.1f} µatm")

## Same query via the data-passport proxy (port 8080)

Reads the stores over HTTP through `run_proxy.py`. Each `.values` access
records chunk fetches; on session close a passport is minted.

The proxy is assumed to run at https://zarr.icos-cp.eu

In [ ]:
import json
from datapassport_zarr import open_zarr

OBSPACK_URL = "https://zarr.icos-cp.eu/icos-obspack.zarr"
FLUXNET_URL = "https://zarr.icos-cp.eu/icos-fluxnet.zarr"

# Same xarray expressions as above — over HTTP, with passports.
# .load() forces the lat/lon coord chunks to fetch eagerly while keeping
# them as xarray DataArrays (with dims), so .where(..., drop=True) accepts
# the boolean mask. Without the eager fetch, lazy reads of lat/lon
# mid-mask can flake on slow / SSH-tunneled connections.
# record_query() captures bbox + surviving stations explicitly so the
# passport JSON shows them.

bbox = {"lat": [LAT_MIN, LAT_MAX], "lon": [LON_MIN, LON_MAX]}

print("=== Obspack — CO2 ===")
with open_zarr(OBSPACK_URL, group="co2", verbose=True) as ds_co2:
    ds_co2.record_query({"bbox": bbox, "time": {"time_co2": [T0, T1]}})
    ds_co2._ds["lat"].load()
    ds_co2._ds["lon"].load()
    co2_nl = (
        ds_co2["co2"]
              .where(
                  (ds_co2.lat >= LAT_MIN) & (ds_co2.lat <= LAT_MAX) &
                  (ds_co2.lon >= LON_MIN) & (ds_co2.lon <= LON_MAX),
                  drop=True,
              )
              .sel(time_co2=slice(T0, T1))
              .compute()
    )
    ds_co2.record_query({"surviving_stations": [
        s.decode() if isinstance(s, (bytes, bytearray)) else str(s)
        for s in co2_nl.station.values
    ]})
    print(f"Obspack CO2 — {len(co2_nl.station)} stations × {len(co2_nl.time_co2)} timestamps")
    print(f"  mean across NL+2024 = {float(co2_nl.mean()):.2f} ppm")

obspack_passport = ds_co2._passport

print("\n=== Fluxnet — monthly NEE ===")
with open_zarr(FLUXNET_URL, group="_combined/fluxnet_mm", verbose=True) as ds_nee:
    ds_nee.record_query({"bbox": bbox, "time": {"time": [T0, T1]},
                         "select": {"ustar_threshold": "VUT", "nee_variant": "REF"}})
    ds_nee._ds["lat"].load()
    ds_nee._ds["lon"].load()
    nee_nl = (
        ds_nee["NEE"]
              .sel(ustar_threshold="VUT", nee_variant="REF")
              .where(
                  (ds_nee.lat >= LAT_MIN) & (ds_nee.lat <= LAT_MAX) &
                  (ds_nee.lon >= LON_MIN) & (ds_nee.lon <= LON_MAX),
                  drop=True,
              )
              .sel(time=slice(T0, T1))
              .compute()
    )
    ds_nee.record_query({"surviving_stations": [
        s.decode() if isinstance(s, (bytes, bytearray)) else str(s)
        for s in nee_nl.station.values
    ]})
    print(f"Fluxnet NEE — {len(nee_nl.station)} stations × {len(nee_nl.time)} months")
    print(f"  mean across NL+2024 = {float(nee_nl.mean()):.3f} {ds_nee['NEE'].attrs.get('units','')}")

fluxnet_passport = ds_nee._passport

print("\n──── Obspack passport ─────────────────────────────────────────────────")
print(json.dumps(obspack_passport, indent=2, default=str))

print("\n──── Fluxnet passport ─────────────────────────────────────────────────")
print(json.dumps(fluxnet_passport, indent=2, default=str))

import pathlib
saved = sorted(pathlib.Path(".passport").glob("*.json"))[-2:] if pathlib.Path(".passport").exists() else []
if saved:
    print(f"\nSaved passport files: {[str(p) for p in saved]}")

### SOCAT via the proxy

One passport per cruise group would be noisy.  Instead we open every
in-bbox cruise inside a single `for`-loop and capture one combined
record on the passport via `record_query`, listing the cruise IDs that
contributed and the bbox/time selection.  Each cruise's
`open_zarr(...).close()` mints its own passport on the proxy side too,
so the chunk-level provenance is preserved.

In [ ]:
import urllib.request

SOCAT_URL = "https://zarr.icos-cp.eu/icos-socat.zarr"

# Discover cruise groups from the proxy's consolidated metadata
with urllib.request.urlopen(f"{SOCAT_URL}/.zmetadata", timeout=30) as r:
    socat_meta_remote = json.load(r).get("metadata") or {}
remote_cruises = sorted({
    k.split("/", 1)[0]
    for k in socat_meta_remote
    if k.endswith("/.zgroup") and k.count("/") == 1
    and not k.startswith(".") and not k.startswith("_")
})

print(f"=== SOCAT via proxy — {len(remote_cruises)} cruise group(s) ===")
contributed: list[str] = []
total_n = 0
sum_weighted = 0.0
all_passports: list[dict] = []

for cruise_id in remote_cruises:
    with open_zarr(SOCAT_URL, group=cruise_id, verbose=False) as ds:
        if "lon" not in ds or "lat" not in ds or "fCO2" not in ds:
            continue
        ds.record_query({"bbox": bbox, "time": [T0, T1], "cruise": cruise_id})
        ds._ds["lon"].load(); ds._ds["lat"].load(); ds._ds["time"].load()
        lon  = ds["lon"].values.astype(float)
        lat  = ds["lat"].values.astype(float)
        fco2 = ds["fCO2"].values.astype(float)
        time = ds["time"].values

        ok = (
            np.isfinite(lon) & np.isfinite(lat) & np.isfinite(fco2)
            & (lat >= LAT_MIN) & (lat <= LAT_MAX)
            & (lon >= LON_MIN) & (lon <= LON_MAX)
            & (time >= np.datetime64(T0)) & (time <= np.datetime64(T1))
        )
        for qc in ("fCO2_QC", "lon_QC", "lat_QC"):
            if qc in ds:
                v = ds[qc].values
                ok &= (v >= 0) & (v <= 2)

        n = int(ok.sum())
        if not n:
            continue
        contributed.append(cruise_id)
        total_n += n
        sum_weighted += float(fco2[ok].mean()) * n
        ds.record_query({"surviving_rows_in_bbox": n})
    if ds._passport:
        all_passports.append({"cruise": cruise_id, "passport_pid": ds._passport.get("passport_pid", "")})

if total_n:
    print(f"  contributed cruises: {len(contributed)}")
    print(f"  rows: {total_n}")
    print(f"  pooled mean fCO2 = {sum_weighted / total_n:.1f} µatm")
else:
    print("  no QC-passing samples in NL+2024.")

print("\n──── SOCAT per-cruise passports ────────────────────────────────────────")
for p in all_passports:
    print(f"  {p['cruise']:14s}  {p['passport_pid'] or '(no PID)'}")

In [ ]:
print("\n──── Last ICOS ocean cruise passport ─────────────────────────────────────────────────")
print(json.dumps(ds._passport, indent=2, default=str))